# EF English Proficiency and ESL Exposure in Europe

This notebook refactors the current analysis to a cleaner, presentation-ready workflow. The goal is to preserve the existing comparison between Eurostat ESL learning indicators and EF EPI outcomes, while removing duplicated preprocessing, debug code, and statistical inconsistency.

In [27]:
import sys
import pathlib

# Ensure the project root is on the notebook path
project_root = pathlib.Path().resolve().parent
sys.path.append(str(project_root))

import eurostat
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

from src.preprocessing import load_ef_epi, clean_eurostat, merge_datasets

## Data loading

Load the Eurostat indicator and the EF EPI dataset. This section preserves the current data sources and keeps the workflow transparent.

In [28]:
datasets = ["educ_uoe_lang01"]
raw_data = {}
for ds in datasets:
    print(f"Loading Eurostat dataset: {ds}")
    raw_data[ds] = eurostat.get_data_df(ds, flags=False)


ef_path = project_root / "data" / "raw" / "efiepi_rankings.csv"
ef = load_ef_epi(str(ef_path))

print("EF dataset shape:", ef.shape)
print("Eurostat dataset shape:", raw_data['educ_uoe_lang01'].shape)

Loading Eurostat dataset: educ_uoe_lang01
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
Country not found: Turkey
EF dataset shape: (238, 7)
Eurostat dataset shape: (12133, 18)


## Preprocessing

This section applies a single, consistent Eurostat cleaning pipeline and merges EF data by ISO code and year. The comparison metric is percentile-based to avoid mixing ranks and z-scores.

In [29]:
euro_df = clean_eurostat(raw_data["educ_uoe_lang01"])
merged = merge_datasets(euro_df, ef)

merged = merged.dropna(subset=["ef_percentile", "learning_percentile"])
merged = merged.sort_values(["year", "iso3"])
latest_year = int(merged["year"].max())
merged_latest = merged[merged["year"] == latest_year]

print("Merged dataset shape:", merged.shape)
print("Latest year used for summary plots:", latest_year)
print(merged[["iso3", "year", "learning", "learning_percentile", "ef_percentile", "gap_pct"]].head())

Merged dataset shape: (835, 7)
Latest year used for summary plots: 2024
    iso3  year  learning  learning_percentile  ef_percentile   gap_pct
610  AUT  2017      99.8             0.806897       0.730769 -0.076127
640  AUT  2017      99.9             0.844828       0.730769 -0.114058
670  AUT  2017      99.2             0.710345       0.730769  0.020424
699  AUT  2017      99.8             0.806897       0.730769 -0.076127
728  AUT  2017      99.0             0.700000       0.730769  0.030769


## Exploratory analysis

The dataset now includes:

- `learning`: Eurostat English learning exposure
- `learning_percentile`: year-wise percentile rank of the ESL measure
- `ef_percentile`: year-wise percentile rank of EF proficiency
- `gap_pct`: the percentile gap between EF proficiency and ESL exposure

In [30]:
summary = merged.groupby("year")[['learning', 'learning_percentile', 'ef_percentile', 'gap_pct']].median()
print(summary)

      learning  learning_percentile  ef_percentile   gap_pct
year                                                        
2017     91.45             0.436207       0.576923  0.096817
2018     93.20             0.449301       0.645161  0.107038
2019     94.80             0.482517       0.666667  0.137762
2020     92.85             0.410345       0.637931  0.165517
2021     94.70             0.434783       0.645161  0.147265
2022     95.40             0.476821       0.633333  0.173510
2023     94.40             0.466216       0.645161  0.220684
2024     94.80             0.465753       0.633333  0.199087


## Europe choropleth: latest-year gap

This static map shows the most recent year’s relative gap between EF proficiency percentiles and ESL exposure percentiles across Europe.

In [31]:
fig = px.choropleth(
    merged_latest,
    locations="iso3",
    color="gap_pct",
    hover_name="geo",
    projection="natural earth",
    color_continuous_scale="RdBu",
    range_color=[-1, 1],
    title=f"Latest-year EF vs ESL gap (percentile) — {latest_year}"
)
fig.update_geos(
    scope="europe",
    showcoastlines=True,
    coastlinecolor="gray",
    showland=True,
    landcolor="lightgray",
    showframe=False
)
fig.update_layout(
    template="plotly_white",
    coloraxis_colorbar_title="gap_pct",
    title_x=0.5,
    margin=dict(l=10, r=10, t=40, b=10)
)
fig.show()


In [35]:
interactive_df = merged.dropna(subset=['gap_pct']).copy()

fig = px.choropleth(
    interactive_df,
    locations="iso3",
    color="gap_pct",
    hover_name="geo",
    animation_frame="year",
    projection="mercator",
    color_continuous_scale="RdYlBu",
    range_color=[-1, 1],
    title="Evolution of English proficiency outcomes relative to ESL exposure across Europe"
)

fig.update_geos(
    scope="europe",
    showcoastlines=True,
    coastlinecolor="gray",
    showland=True,
    landcolor="lightgray",
    showframe=False,
    showcountries=True
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    coloraxis_colorbar_title="gap_pct",
    margin=dict(l=10, r=10, t=50, b=10),
    width=950,
    height=650
)

fig.show()

## Italy vs Europe average trajectory

This figure compares Italy’s annual percentile trajectory with the European average for the same measures.


In [33]:
italy_traj = (
    merged[merged["iso3"] == "ITA"]
    .groupby("year", as_index=False)["gap_pct"]
    .mean()
)

europe_avg = (
    merged
    .groupby("year", as_index=False)["gap_pct"]
    .mean()
    .rename(columns={"gap_pct": "europe_gap_pct"})
)

traj = pd.merge(italy_traj, europe_avg, on="year")

fig = go.Figure()

# Europe reference line
fig.add_trace(
    go.Scatter(
        x=traj["year"],
        y=traj["europe_gap_pct"],
        mode="lines",
        name="Europe average",
        line=dict(color="gray", width=2, dash="dash")
    )
)

# Italy focus line
fig.add_trace(
    go.Scatter(
        x=traj["year"],
        y=traj["gap_pct"],
        mode="lines+markers",
        name="Italy",
        line=dict(color="#d62728", width=4),
        marker=dict(size=8)
    )
)

# Baseline
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black"
)

fig.update_layout(
    template="plotly_white",
    title="Italy consistently underperforms relative to ESL exposure",
    title_x=0.5,
    xaxis_title="Year",
    yaxis_title="Relative performance gap",
    width=950,
    height=550,
    margin=dict(l=40, r=40, t=60, b=40),
    legend_title=""
)

fig.show()


## Interpretation and limitations

The final analysis uses percentile ranks for both ESL exposure and EF proficiency. This makes the gap metric coherent and easier to interpret. The current work is intentionally descriptive and avoids complex lag modelling or causal claims.

Limitations:

- The gap metric is relative within each year, not an absolute measure.
- Country coverage depends on ISO harmonization and available Eurostat/EPI observations.
- A full dashboard can be added later once the cleaned analytical workflow is stable.